### nxsut generator — v3.0

MARIO-native pipeline: parses EXIOBASE Hybrid v3.3.18, updates the electricity supply mix from EMBER (via the nxbase query API), pools electricity trade behind a supply/need pass-through layer, and updates the trade mix from the **open ENTSO-E** scheduled-exchange set (via the nxbase query API). Fully open input chain — the first publishable nxsut version. Supersedes the retired `v2.1` (which used the proprietary Electricity Maps mix on the same MARIO-native pipeline).

ENTSO-E covers the European countries; every other EXIOBASE region (US, CN, JP, … and the RoW aggregates) is filled domestic-only — as are the origin-only regions ENTSO-E returns as an all-zero destination column (e.g. Luxembourg, Malta: their real import dependence disappears in this version, a known residual pending an ENTSO-E control-area fetch).

Set `user` and `year` in the first cell, then run top to bottom.

In [ ]:
import mario
import yaml
import os

pfile = 'paths_personal.yml' if os.path.exists('paths_personal.yml') else 'paths.yml'
with open(pfile, 'r') as file:  # personal override (git-ignored), else the template
    paths = yaml.safe_load(file)

user = 'USER'   # your key in paths(_personal).yml
year = 2023   # change this to the year you want to build

paths = paths[user]
import warnings
warnings.filterwarnings("ignore")

from support import nxbase_client as nxc
nxbase_api = paths.get('nxbase_api', nxc.DEFAULT_API)


Parse the raw EXIOBASE database, aggregate electricity to EMBER resolution, then add the explicit steel & H2 production routes (Ghezzi et al. 2026 recipe, fetched from the nxbase query API). `add_sectors` runs **after** `aggregate_ee`: the routes attach to the single aggregated grid `Electricity`, and the pre-existing EMBER electricity activities keep their supply coefficients. `meta.source` enables MARIO's EXIOBASE Rest-of-World member-country expansion when using EMBER.

In [ ]:
db = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')
db.meta.source = 'EXIOBASE Hybrid 3.3.18'
db.aggregate('support/aggregate_ee.xlsx', ignore_nan=True)

# steel/H2 sectors from the nxbase recipe -- add AFTER aggregate_ee so the new
# routes attach to the aggregated grid 'Electricity', and the pre-existing EMBER
# electricity activities keep their supply coefficients (running add_sectors
# before aggregate would drop them from the 's' block -> update_supply_mix fails).
nxc.build_add_sectors_master('support/add_sectors/Master_steel_h2.xlsx', '_steel_master.xlsx', api_url=nxbase_api)
db.read_add_sectors_excel('_steel_master.xlsx', read_inventories=True)
db.add_sectors()

### Furnace-gas emission reallocation (ExioSteel method)

Two fictitious activities take the blast/oxygen furnace gas by-products (the steel sector's supply of them is zeroed), and the steel sector's coefficients are recomputed on its steel supply alone (`U/S_main`, not `U/X`) → footprint per tonne of *actual steel*, not diluted by the co-product gases. Runs **after** the Ghezzi `add_sectors`, **before** the supply mix.

In [ ]:
# Furnace-gas emission reallocation (ExioSteel method): two fictitious gas-
# production activities take the blast/oxygen furnace gas by-products (steel
# supply of them zeroed), and the steel sector's U/V/E are recomputed on its
# steel supply alone (/ S_main, not / X) -> footprint per tonne of actual steel.
db.read_add_sectors_excel('support/add_sectors/blastfurnacegas.xlsx', read_inventories=True)
db.add_sectors()

STEEL_ACT = 'Manufacture of basic iron and steel and of ferro-alloys and first products thereof'
STEEL_COM = 'Basic iron and steel and of ferro-alloys and first products thereof'
s, u, v, e = db.s, db.u, db.v, db.e
by_product = list(db.add_sectors_master['Commodity'].unique())
gas_of_act = {a: db.add_sectors_master.loc[db.add_sectors_master['Activity'] == a, 'Commodity'].values[0]
              for a in db.new_activities}
for region in db.get_index('Region'):
    s_byprod = (s.loc[(region, 'Activity', STEEL_ACT), (region, 'Commodity', by_product)] * 0).to_frame().T
    s.update(s_byprod)
    for new_act, commodity in gas_of_act.items():
        s.loc[(region, 'Activity', new_act), (region, 'Commodity', commodity)] = 1
    S_main = db.S.loc[(region, 'Activity', STEEL_ACT), (region, 'Commodity', STEEL_COM)]
    u.update(db.U.loc[:, (region, 'Activity', STEEL_ACT)] / S_main.sum())
    v.update(db.V.loc[:, (region, 'Activity', STEEL_ACT)] / S_main.sum())
    e.update(db.E.loc[:, (region, 'Activity', STEEL_ACT)] / S_main.sum())

z = db.z
z.update(s)
z.update(u)
db.update_scenarios('baseline', z=z, v=v, e=e)
db.reset_to_coefficients('baseline')
print('BFG/OFG reallocation applied; new activities:', list(db.new_activities))

Supply mix from nxbase (query API) — see `support/nxbase_client.py`. MARIO reads the reduced EMBER snapshot from a transient file, regenerated every run.

In [ ]:
from support import nxbase_client as nxc

nxbase_api = paths.get('nxbase_api', nxc.DEFAULT_API)
print(nxc.get_provenance(nxbase_api, ['EMBER Yearly Electricity Data 2025']))

ember_snapshot_path = 'support/_nxbase_ember_snapshot.csv'
nxc.get_ember_snapshot(nxbase_api).to_csv(ember_snapshot_path, index=False)

db.update_supply_mix(
    "electricity",
    scenario = 'baseline',
    year = year,
    ember_path = ember_snapshot_path,
)

Pool the trade of the selected commodities. MARIO adds the `" supply"` / `" need"` pass-through layer and stores the observed trade shares in the supply block market shares. The suffixes match the `NXS2` namespace rows in nxbase.

In [ ]:
traded_commodities = ['Electricity']
db.pool_trade(traded_commodities, supply_suffix=" supply", need_suffix=" need")

**Update trade mixes** from the open ENTSO-E scheduled-exchange set (nxbase query API). One origins-by-destinations matrix per commodity; a positive column sum — not mere presence — decides "covered" (ENTSO-E returns some destinations as an all-zero column, which `update_trade_mix` would otherwise reject). `rescale=True` normalizes each destination mix while preserving destination column totals.

In [ ]:
scenario = 'entsoe_trades'
if scenario not in db.scenarios:
    db.clone_scenario('baseline', scenario)

regions = list(db.get_index('Region'))
for commodity in traded_commodities:
    pooled = db.meta.pooled_trade_map[commodity]
    trades = nxc.get_trade_matrix(
        nxbase_api, year=year, commodity=commodity,
        source=f"ENTSO-E electricity import mix {year}",
    )
    trade_dict = {}
    for dest in regions:
        col = trades[dest] if dest in trades.columns else None
        trade_dict[dest] = (
            col.dropna().to_dict() if (col is not None and col.sum() > 0) else {dest: 1.0}
        )
    db.update_trade_mix(
        trade_dict,
        items = pooled['supply'],
        commodities = pooled['need'],
        scenario = scenario,
        rescale = True,
    )

Export v3.0.

In [ ]:
v30_path = os.path.join(paths['export'], "v3.0", str(year))
os.makedirs(v30_path, exist_ok=True)
db.to_txt(path = v30_path, scenario = scenario)